# Exercise 3 — Image Retrieval Using Embeddings

**Course:** CAS Deep Learning — Computer Vision  
**Day:** 2 | **Block:** 3

## Learning objectives
After this exercise you should be able to:
- Extract embedding vectors from a pretrained vision model
- Use cosine similarity to find nearest neighbours in embedding space
- Build a simple image retrieval pipeline
- Compare embedding quality between a supervised and a self-supervised model

## Instructions
- Run cells top to bottom.
- Fill in code where you see `# YOUR CODE HERE`.
- Estimated time: ~35–45 minutes.

In [ ]:
import sys

import pyrootutils

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q -r requirements.txt
    from google.colab import drive

    ROOT_PATH = "/content/drive"
    drive.mount(ROOT_PATH)
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")
DATA_PATH.mkdir(parents=True, exist_ok=True)

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Imports

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm
from transformers import AutoImageProcessor, AutoModel

## Load the Product Dataset

We use the **ABO Furniture** dataset — a curated subset of the Amazon Berkeley Objects dataset. It contains product images across 6 furniture categories: bed, chair, lamp, sofa, storage, and table.

Unlike classification, our goal here is not to predict a label. Instead, given a query image, we want to find the most **visually similar** products — this is **image retrieval**.

In [ ]:
def load_ecommerce_dataset(data_path):
    """Download and extract the ABO furniture subset."""
    from pathlib import Path

    repo_id = "marco-willi/ecommerce-abo-furniture"
    filename = "abo_furniture.tar.gz"
    dataset_dir = Path(data_path) / "abo_furniture"

    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    print(f"Downloading ABO furniture from HF Hub ({repo_id}) ...")
    import tarfile

    from huggingface_hub import hf_hub_download

    archive = hf_hub_download(repo_id, filename, repo_type="dataset")
    print(f"Extracting to {data_path} ...")
    with tarfile.open(archive) as tar:
        tar.extractall(data_path)

    assert dataset_dir.exists(), f"Extraction failed — {dataset_dir} not found"
    print(f"Ready: {dataset_dir}")
    return dataset_dir


DATASET_PATH = load_ecommerce_dataset(DATA_PATH)
print(f"\nDataset path: {DATASET_PATH}")

In [ ]:
# Load the gallery and inspect
gallery_raw = ImageFolder(root=DATASET_PATH / "val")
class_names = gallery_raw.classes

print(f"Gallery images: {len(gallery_raw)}")
print(f"Categories ({len(class_names)}): {class_names}")

In [ ]:
# Show one random sample per category
random.seed(123)

_fig, axes = plt.subplots(1, len(class_names), figsize=(18, 3))

for idx, cls_name in enumerate(class_names):
    cls_indices = [i for i, (_, label) in enumerate(gallery_raw.imgs) if label == idx]
    sample_idx = random.choice(cls_indices)
    image, _ = gallery_raw[sample_idx]

    _ = axes[idx].imshow(image)
    _ = axes[idx].set_title(cls_name, fontsize=12)
    _ = axes[idx].axis("off")

_ = plt.suptitle("ABO Furniture — One Sample Per Category", fontsize=14)
_ = plt.tight_layout()
plt.show()

## Part 1 — Extract Image Embeddings

Pretrained vision models do more than classify images — their internal representations capture rich visual information. By extracting these representations (called **embeddings**), we can compare images based on visual similarity rather than class labels.

We start with **DINOv2**, a self-supervised model trained by Meta. Unlike supervised models (trained with labels), DINOv2 learned visual representations purely from unlabelled images — making its embeddings particularly good for general-purpose similarity tasks.

In [ ]:
# Load the DINOv2-small model and its image processor
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino_model = AutoModel.from_pretrained("facebook/dinov2-small")
dino_model = dino_model.eval().to(device)

print(f"Model: {dino_model.__class__.__name__}")
print(f"Embedding dimension: {dino_model.config.hidden_size}")

DINOv2 is a Vision Transformer. It splits each image into patches and processes them through transformer layers. The output includes a special **`[CLS]` token** — a single vector that summarizes the entire image. This is our embedding.

Extract the `[CLS]` token embedding for the first image in the gallery:
1. Load the image from `gallery_raw[0]`
2. Preprocess it using `processor(images=image, return_tensors="pt")`
3. Pass through the model (use `torch.no_grad()`)
4. Get the `[CLS]` token: `outputs.last_hidden_state[:, 0, :]`
5. Store the result in a variable called `embedding`

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert embedding.shape == (1, 384), f"Expected (1, 384), got {embedding.shape}"
assert embedding.dtype == torch.float32
print("OK")

**Question:** The embedding is a 384-dimensional vector. What kind of information does it encode?

<details>
<summary>Click to reveal answer</summary>

The embedding captures **high-level visual features** of the image — shape, colour, texture, spatial structure, and object identity. Unlike raw pixels (which encode low-level colour values), embeddings are **semantic**: two images of different chairs will have similar embeddings, even if their pixel values are completely different.

DINOv2 embeddings are especially good at capturing semantic similarity because the model was trained with self-supervised objectives that encourage consistent representations across different views of the same scene.

</details>

## Part 2 — Build an Embedding Database

To retrieve similar images, we need embeddings for **all** gallery images — not just one. We build an embedding database by running every image through the model and stacking the results into a matrix of shape `(N, D)`, where `N` is the number of images and `D` is the embedding dimension.

For efficiency, we process images in batches using a `DataLoader`.

In [ ]:
# Define transforms matching what DINOv2 expects
# (same ImageNet preprocessing as Exercises 1 and 2)
val_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Create a transformed version of the gallery for model input
gallery_dataset = ImageFolder(root=DATASET_PATH / "val", transform=val_transforms)

# Use a random subset to keep extraction fast on CPU
N_GALLERY = min(500, len(gallery_dataset))
random.seed(42)
gallery_indices = random.sample(range(len(gallery_dataset)), N_GALLERY)
gallery_subset = Subset(gallery_dataset, gallery_indices)

gallery_loader = DataLoader(gallery_subset, batch_size=32, shuffle=False, num_workers=0)

print(f"Gallery size: {len(gallery_subset)} images")
print(f"Batches: {len(gallery_loader)}")

Extract DINOv2 embeddings for all gallery images. Loop over `gallery_loader`, pass each batch through `dino_model`, and collect the `[CLS]` token embeddings.

Store the results in:
- `dino_embeddings` — tensor of shape `(N_GALLERY, 384)`
- `gallery_labels` — list of integer labels

Hints:
- Use `dino_model(pixel_values=images)` to pass batches from the DataLoader
- Extract the `[CLS]` token: `outputs.last_hidden_state[:, 0, :]`
- Collect embeddings in a list, then `torch.cat()` at the end

<details>
<summary>Click to reveal solution</summary>

```python
all_embeddings = []
all_labels = []

for images, labels in tqdm(gallery_loader, desc="DINOv2"):
    images = images.to(device)
    with torch.no_grad():
        outputs = dino_model(pixel_values=images)
    embs = outputs.last_hidden_state[:, 0, :]
    all_embeddings.append(embs.cpu())
    all_labels.extend(labels.tolist())

dino_embeddings = torch.cat(all_embeddings)
gallery_labels = all_labels
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert dino_embeddings.shape == (N_GALLERY, 384), (
    f"Expected ({N_GALLERY}, 384), got {dino_embeddings.shape}"
)
assert len(gallery_labels) == N_GALLERY

# Normalize embeddings to unit length (L2 norm)
dino_embeddings = F.normalize(dino_embeddings, p=2, dim=1)

# Verify normalization
norms = dino_embeddings.norm(dim=1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)
print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}")
print("OK")

**Question:** Why do we normalize the embeddings to unit length before computing similarity?

<details>
<summary>Click to reveal answer</summary>

We normalize so that **cosine similarity reduces to a simple dot product**:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$

When both vectors have unit length ($\|a\| = \|b\| = 1$), this simplifies to $a \cdot b$.

Normalization also ensures that **similarity depends on direction, not magnitude**. Without it, a longer embedding vector could appear more similar simply because of its larger magnitude, not because it represents a more similar image.

</details>

## Part 3 — Nearest Neighbour Retrieval

With all gallery images embedded, retrieval is straightforward:
1. Compute the embedding of the query image
2. Compute cosine similarity between the query and every gallery embedding
3. Return the gallery images with the highest similarity

Since our embeddings are L2-normalized, cosine similarity is just a matrix multiplication.

Implement `retrieve_similar(query_idx, embeddings, top_k=5)` that:
1. Takes the query embedding from `embeddings[query_idx]`
2. Computes cosine similarity to all other embeddings (use `@` for matrix multiply)
3. Returns the indices and similarity scores of the `top_k` most similar images (excluding the query itself)

Return a tuple `(indices, scores)` where both are 1-D tensors of length `top_k`.

<details>
<summary>Click to reveal solution</summary>

```python
def retrieve_similar(query_idx, embeddings, top_k=5):
    query = embeddings[query_idx]
    similarities = embeddings @ query
    similarities[query_idx] = -1  # exclude query itself
    scores, indices = similarities.topk(top_k)
    return indices, scores
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify: retrieve top-5 for the first image
test_indices, test_scores = retrieve_similar(0, dino_embeddings, top_k=5)
assert test_indices.shape == (5,), f"Expected 5 indices, got {test_indices.shape}"
assert test_scores.shape == (5,), f"Expected 5 scores, got {test_scores.shape}"
assert 0 not in test_indices, "Query index should be excluded from results"
assert (test_scores[:-1] >= test_scores[1:]).all(), "Scores should be sorted descending"
print(f"Top-5 indices: {test_indices.tolist()}")
print(f"Top-5 scores:  {[f'{s:.3f}' for s in test_scores.tolist()]}")
print("OK")

Let's see retrieval in action. For 3 query images from different categories, we retrieve the top-5 most similar products and display them side by side.

In [ ]:
def show_retrieval(
    query_idx, embeddings, gallery_raw, gallery_indices, class_names, top_k=5
):
    """Display a query image and its top-k retrieved results."""
    indices, scores = retrieve_similar(query_idx, embeddings, top_k)

    _fig, axes = plt.subplots(1, 1 + top_k, figsize=(3 * (1 + top_k), 3.5))

    # Show query
    q_raw_idx = gallery_indices[query_idx]
    q_image, q_label = gallery_raw[q_raw_idx]
    _ = axes[0].imshow(q_image)
    _ = axes[0].set_title(
        f"QUERY\n{class_names[q_label]}", fontsize=11, fontweight="bold", color="blue"
    )
    _ = axes[0].axis("off")

    # Show retrieved images
    for rank, (ret_idx, score) in enumerate(zip(indices, scores, strict=True)):
        raw_idx = gallery_indices[ret_idx.item()]
        ret_image, ret_label = gallery_raw[raw_idx]
        ax = axes[rank + 1]
        _ = ax.imshow(ret_image)
        match = "green" if ret_label == q_label else "red"
        _ = ax.set_title(
            f"#{rank + 1} ({score:.2f})\n{class_names[ret_label]}",
            fontsize=10,
            color=match,
        )
        _ = ax.axis("off")

    _ = plt.tight_layout()
    return plt.show()

In [ ]:
# Pick 3 query images from different categories
query_indices = []
seen_labels = set()
for i, label in enumerate(gallery_labels):
    if label not in seen_labels:
        query_indices.append(i)
        seen_labels.add(label)
    if len(query_indices) == 3:
        break

for q_idx in query_indices:
    fig = show_retrieval(
        q_idx, dino_embeddings, gallery_raw, gallery_indices, class_names
    )
    plt.show()

**Question:** Look at the retrieved results. Are the top matches from the same category as the query? Do they also look visually similar in terms of style, shape, or colour?

<details>
<summary>Click to reveal answer</summary>

DINOv2 typically retrieves products that are both **semantically similar** (same category — chairs retrieve chairs, lamps retrieve lamps) and **visually similar** (similar shape, colour, and style). This goes beyond simple category matching: a modern minimalist chair tends to retrieve other modern minimalist chairs, not ornate antique ones.

This is because self-supervised training encourages the model to capture fine-grained visual structure, not just category-level features. The model was never trained with furniture labels — it discovered these visual similarities on its own.

</details>

Let's try a few more queries, including some that might be more challenging.

In [ ]:
# Show 3 more random queries
random.seed(123)
extra_queries = random.sample(range(N_GALLERY), 3)

for q_idx in extra_queries:
    fig = show_retrieval(
        q_idx, dino_embeddings, gallery_raw, gallery_indices, class_names
    )
    plt.show()

## Part 4 — Compare with a Supervised Model

DINOv2 was trained **without labels** (self-supervised). How does it compare to a model trained **with labels** (supervised)? We extract embeddings from a ResNet-50 pretrained on ImageNet classification and compare the retrieval quality.

**Key difference:**
- **ResNet-50 (supervised)** — trained to distinguish 1,000 ImageNet categories. Its features are optimized for classification, not similarity.
- **DINOv2 (self-supervised)** — trained to produce consistent representations across views of the same image. Its features are optimized for general visual understanding.

In [ ]:
# Load ResNet-50 in feature-extraction mode (no classification head)
resnet_model = timm.create_model("resnet50", pretrained=True, num_classes=0)
resnet_model = resnet_model.eval().to(device)

# num_classes=0 removes the classification head and returns
# the global average pooled features directly
dummy = torch.randn(1, 3, 224, 224, device=device)
with torch.no_grad():
    dummy_out = resnet_model(dummy)
print(f"ResNet-50 embedding dimension: {dummy_out.shape[1]}")

Extract ResNet-50 embeddings for the same gallery images. Loop over `gallery_loader`, pass each batch through `resnet_model`, and collect the output embeddings.

Store the result in `resnet_embeddings` — a tensor of shape `(N_GALLERY, 2048)`.

Hint: ResNet-50 with `num_classes=0` directly returns the feature vector — no need to index into `last_hidden_state`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert resnet_embeddings.shape == (N_GALLERY, 2048), (
    f"Expected ({N_GALLERY}, 2048), got {resnet_embeddings.shape}"
)

# Normalize ResNet embeddings
resnet_embeddings = F.normalize(resnet_embeddings, p=2, dim=1)
print("OK")

Now we compare: for the same query images, how do DINOv2 and ResNet-50 retrieval results differ?

In [ ]:
# Side-by-side comparison: DINOv2 vs ResNet-50
TOP_K = 5

for q_idx in query_indices:
    q_raw_idx = gallery_indices[q_idx]
    q_image, q_label = gallery_raw[q_raw_idx]

    dino_idx, dino_scores = retrieve_similar(q_idx, dino_embeddings, TOP_K)
    resnet_idx, resnet_scores = retrieve_similar(q_idx, resnet_embeddings, TOP_K)

    _fig, axes = plt.subplots(2, 1 + TOP_K, figsize=(3 * (1 + TOP_K), 7))

    # Row 0: DINOv2
    _ = axes[0, 0].imshow(q_image)
    _ = axes[0, 0].set_title(
        f"QUERY\n{class_names[q_label]}", fontsize=10, fontweight="bold", color="blue"
    )
    _ = axes[0, 0].axis("off")
    _ = axes[0, 0].set_ylabel(
        "DINOv2", fontsize=12, fontweight="bold", rotation=0, labelpad=50
    )

    for rank, (ret_idx, score) in enumerate(zip(dino_idx, dino_scores, strict=True)):
        raw_idx = gallery_indices[ret_idx.item()]
        ret_image, ret_label = gallery_raw[raw_idx]
        match = "green" if ret_label == q_label else "red"
        _ = axes[0, rank + 1].imshow(ret_image)
        _ = axes[0, rank + 1].set_title(
            f"#{rank + 1} ({score:.2f})\n{class_names[ret_label]}",
            fontsize=9,
            color=match,
        )
        _ = axes[0, rank + 1].axis("off")

    # Row 1: ResNet-50
    _ = axes[1, 0].imshow(q_image)
    _ = axes[1, 0].set_title(
        f"QUERY\n{class_names[q_label]}", fontsize=10, fontweight="bold", color="blue"
    )
    _ = axes[1, 0].axis("off")
    _ = axes[1, 0].set_ylabel(
        "ResNet-50", fontsize=12, fontweight="bold", rotation=0, labelpad=50
    )

    for rank, (ret_idx, score) in enumerate(
        zip(resnet_idx, resnet_scores, strict=True)
    ):
        raw_idx = gallery_indices[ret_idx.item()]
        ret_image, ret_label = gallery_raw[raw_idx]
        match = "green" if ret_label == q_label else "red"
        _ = axes[1, rank + 1].imshow(ret_image)
        _ = axes[1, rank + 1].set_title(
            f"#{rank + 1} ({score:.2f})\n{class_names[ret_label]}",
            fontsize=9,
            color=match,
        )
        _ = axes[1, rank + 1].axis("off")

    _ = plt.tight_layout()
plt.show()

**Question:** Which model retrieves more visually similar products? Why might self-supervised pretraining help for retrieval?

<details>
<summary>Click to reveal answer</summary>

**DINOv2 typically produces better retrieval results** — the retrieved images tend to be more visually coherent (similar style, shape, and colour) rather than just from the same broad category.

Why? The two models were trained with fundamentally different objectives:

- **ResNet-50 (supervised on ImageNet)** was trained to separate 1,000 object categories. Its embeddings are optimized for *classification boundaries* — it knows that chairs are different from tables, but it does not need to distinguish between styles of chairs. Two very different-looking chairs might have similar embeddings simply because they are both "chairs."

- **DINOv2 (self-supervised)** was trained to produce *consistent representations across different views* of the same image. This forces the model to capture fine-grained visual structure — shape, material, colour, style — because it needs to recognize that two augmented views of the same chair represent the same object.

This makes self-supervised models particularly well-suited for **retrieval**, where fine-grained similarity matters more than categorical boundaries.

</details>

Let's quantify the difference. We compute how often each model's top-5 results contain images from the same category as the query.

In [ ]:
# Category-match precision@5 for both models
def precision_at_k(embeddings, labels, k=5):
    """Fraction of top-k results that share the query's category."""
    correct = 0
    total = 0
    for q_idx in range(len(labels)):
        indices, _ = retrieve_similar(q_idx, embeddings, top_k=k)
        q_label = labels[q_idx]
        matches = sum(1 for idx in indices if labels[idx.item()] == q_label)
        correct += matches
        total += k
    return correct / total


dino_p5 = precision_at_k(dino_embeddings, gallery_labels, k=5)
resnet_p5 = precision_at_k(resnet_embeddings, gallery_labels, k=5)

print("Category-match Precision@5:")
print(f"  DINOv2:    {dino_p5:.1%}")
print(f"  ResNet-50: {resnet_p5:.1%}")

## Part 5 — Visualize the Embedding Space

Embeddings live in a high-dimensional space (384 or 2048 dimensions). We can project them down to 2D using **PCA** to see how images cluster.

In [ ]:
from sklearn.decomposition import PCA

# Project DINOv2 embeddings to 2D
pca = PCA(n_components=2)
dino_2d = pca.fit_transform(dino_embeddings.numpy())

_fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DINOv2 embedding space
for label_idx, name in enumerate(class_names):
    mask = np.array(gallery_labels) == label_idx
    if mask.sum() > 0:
        _ = axes[0].scatter(
            dino_2d[mask, 0], dino_2d[mask, 1], label=name, alpha=0.6, s=20
        )
_ = axes[0].set_title("DINOv2 Embeddings (PCA)", fontsize=12)
_ = axes[0].legend(fontsize=9)
_ = axes[0].set_xlabel("PC 1")
_ = axes[0].set_ylabel("PC 2")

# ResNet-50 embedding space
resnet_2d = PCA(n_components=2).fit_transform(resnet_embeddings.numpy())
for label_idx, name in enumerate(class_names):
    mask = np.array(gallery_labels) == label_idx
    if mask.sum() > 0:
        _ = axes[1].scatter(
            resnet_2d[mask, 0], resnet_2d[mask, 1], label=name, alpha=0.6, s=20
        )
_ = axes[1].set_title("ResNet-50 Embeddings (PCA)", fontsize=12)
_ = axes[1].legend(fontsize=9)
_ = axes[1].set_xlabel("PC 1")
_ = axes[1].set_ylabel("PC 2")

_ = plt.suptitle("Embedding Space: Do Categories Form Clusters?", fontsize=14)
_ = plt.tight_layout()
plt.show()

**Question:** How do the cluster structures compare between DINOv2 and ResNet-50?

<details>
<summary>Click to reveal answer</summary>

Both models group images by category, but the **cluster quality** often differs:

- **DINOv2** tends to produce tighter, more separated clusters. Categories are well-grouped, and within each cluster, visually similar items (e.g., similar chair styles) appear close together.
- **ResNet-50** also groups by category, but the clusters may overlap more — especially for categories that share visual features (e.g., tables and desks, or beds and sofas).

Remember: PCA only shows the top-2 principal components. The full 384/2048-dimensional space may separate categories better than the 2D projection suggests.

</details>

## Follow-up — Try a Different Domain (Open-Ended)

The retrieval pipeline we built is domain-agnostic — it works for any images, not just furniture. A natural question is: **how well does it generalize to a completely different product domain?**

The **DeepFashion In-Shop** dataset contains thousands of clothing images (tops, dresses, pants, etc.) with multiple views per garment. This makes it ideal for testing retrieval: can the model find the same garment photographed from a different angle?

If you have access to the DeepFashion dataset (ask your instructor for the Google Drive file ID), try running the same pipeline on fashion images.

In [ ]:
# DeepFashion download helper — only works with instructor-provided file ID
def load_deepfashion(data_path, gdrive_file_id=None):
    """Load the DeepFashion classroom subset (internal, restricted distribution)."""
    from pathlib import Path

    dataset_dir = Path(data_path) / "deepfashion"

    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    if gdrive_file_id is None:
        raise FileNotFoundError(
            "DeepFashion dataset not found locally.\n"
            "Ask your instructor for the Google Drive file ID and pass it as\n"
            "  load_deepfashion(data_path, gdrive_file_id='<ID>')"
        )

    import zipfile

    import gdown

    zip_path = Path(data_path) / "deepfashion_classroom_v1_internal.zip"
    url = f"https://drive.google.com/uc?id={gdrive_file_id}"

    print("Downloading DeepFashion classroom subset from Google Drive ...")
    gdown.download(url, str(zip_path), quiet=False)

    print(f"Extracting to {data_path} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_path)

    zip_path.unlink(missing_ok=True)

    assert dataset_dir.exists(), f"Extraction failed — {dataset_dir} not found"
    print(f"Ready: {dataset_dir}")
    return dataset_dir


# Uncomment and fill in the file ID to try DeepFashion:
# DEEPFASHION_PATH = load_deepfashion(DATA_PATH, gdrive_file_id="YOUR_ID_HERE")

### Open-ended tasks

If you have access to the DeepFashion dataset, try:

1. **Load the fashion dataset** using `load_deepfashion()` and create a gallery from the val split
2. **Extract DINOv2 embeddings** for the fashion gallery (reuse the same pipeline)
3. **Query a dress image** — does the model retrieve other dresses? Are they visually similar in style?
4. **Cross-domain query** — what happens if you embed a furniture image and query it against the fashion gallery? What does the model return?

These experiments test how general DINOv2's representations really are.

## Summary

In this exercise you built an image retrieval system from scratch:

1. **Embeddings** — extracted 384-dim vectors from DINOv2 that capture the visual content of each image
2. **Similarity search** — used cosine similarity to find nearest neighbours in embedding space
3. **Retrieval pipeline** — query an image, retrieve the top-k most similar products
4. **Model comparison** — DINOv2 (self-supervised) vs. ResNet-50 (supervised) — self-supervised embeddings often capture finer-grained visual similarity

**Key takeaways:**
- Pretrained models are powerful feature extractors, even without task-specific training
- Self-supervised models like DINOv2 learn representations that transfer well to similarity tasks
- The same pipeline works across different domains (furniture, fashion, wildlife) — embeddings are general-purpose

Next: adapting pretrained models to new tasks with k-NN, linear probes, and fine-tuning!